# infra-defect-detection — Phase 3 on Kaggle(跨国分布漂移诊断)

**重要:跑这个notebook之前,先在右侧 Session options / Settings 里把 Accelerator 切成 GPU(T4 x2 或 P100)**——默认是CPU-only。

对应路线图阶段3:phase 2在单一国家(Czech)内建立了"同国家训练/测试"基线(mAP@50=0.3113)。phase 3要回答的问题是——**换成没在训练里见过的国家,掉多少、为什么掉**。做法:

- **训练国家**(合并成一个train/val集):Japan、India、Czech——三国合并后数据量最大,且把phase 2已经用过的Czech保留在训练集里,不浪费。
- **目标国家**(完全不参与训练,各自单独评测,不合并成一个数字):Norway、United_States、China_MotorBike、China_Drone——分开报告是因为China的摩托车拍摄和无人机拍摄本身就是两种不同的设备/高度条件,合并成一个"China"数字会掩盖到底是哪一个在拖后腿。

产出要求(不是只报一个"掉了多少mAP"):
- 同一个模型,在**in-domain held-out test**(训练国家里切出来、从没训练过的部分,公平基线)和**每个目标国家**上分别评测
- 每个目标国家的具体失败案例(漏检/多检/分类错误)标注图,不只是一个数字
- 失败案例的亮度等线索,方便和phase 1 EDA报告里各国亮度对比,判断是不是光照/天气差异在主导

## 目录
0. 写入脚本(phase 1/2的4个脚本 + phase 3新增的3个:make_cross_country_split.py、train_cross_country_baseline.py、diagnose_failures.py)
1. 安装依赖
2. 下载 + MD5校验(优先用下面的Dataset缓存,没有才走FigShare)
3. 解压转换**全部7个国家**(优先直接用Dataset缓存里已经解压好的数据,没有才走zip)
4. 生成跨国split(source合并70/15/15,target各自全量做eval list)
5. 训练(一次)+ 在in-domain test和4个target国家上分别评测——**这一步比phase 2慢很多**(见下面的时间预估)
6. 失败案例诊断:每个target国家挑最差的几张图,画出GT(蓝)和预测(红)框
7. 核对结果
8. 收尾:全部打包成一个zip,一次性下载(phase 2下11个文件太麻烦,这次直接打包)


## 在跑之前:先挂载缓存Dataset(强烈建议,能省下12.35GB下载 + 全量解压的时间)

之前专门跑了一个一次性的"cache builder" notebook,把FigShare的合并包下载、MD5校验通过后做成了一个私有Kaggle Dataset——`rdd2022-figshare-zip-cache`。**建过程中发现Kaggle的"New Dataset from notebook output"会自动把output里的zip递归解压**(不只是外层合并包,连里面7个国家各自的nested zip也一起展开了),所以这个Dataset实际存的不是一个zip,而是**已经解压好、可以直接转换的7个国家的原始VOC数据**——比缓存一个zip还省事,连解压这一步都不用再跑一遍。

**操作**(在打开这个notebook之后、跑下面任何代码格之前):
1. 右侧栏找到 **Add Input**(或者顶部菜单 File → Add Input)
2. 搜索 `rdd2022-figshare-zip-cache`,点击右侧的 `+` 把它加进来
3. 加完之后应该能在 `/kaggle/input/rdd2022-figshare-zip-cache/` 下看到数据——不用手动做任何事,下面`download_rdd2022.py`和`extract_convert_per_country.py`都会自动检测`/kaggle/input/`下面有没有这个缓存,有就直接用(跳过下载和解压),没有就照常走FigShare下载+本地解压转换那条路(慢,但不会报错中断)。

如果不加这个Dataset也完全没问题,只是第2、3步会变慢很多(要下载12.35GB + 全量解压全部7个国家)。

## 0. 写入脚本

In [ ]:
import os
os.makedirs("scripts", exist_ok=True)
os.makedirs("data", exist_ok=True)


In [ ]:
%%writefile scripts/download_rdd2022.py
"""Download and extract the RDD2022 road damage dataset (roadmap: infra-defect-detection, phase 1).

RDD2022 covers six countries (Japan, India, Czech Republic, Norway, United States, China) with
47,420 road images and 55,000+ annotated damage instances across four classes (D00 longitudinal
crack, D10 transverse crack, D20 alligator crack, D40 pothole). Images are CC BY-SA 4.0 per the
sekilab/RoadDamageDetector GitHub README (attribute sekilab/RoadDamageDetector + the RDD2022 paper,
arxiv.org/abs/2209.08538, in any README/Model Card that uses this data) - note the FigShare listing
below shows "CC BY 4.0" for the same dataset; this discrepancy between the two official sources is
unresolved, so treat CC BY-SA 4.0 (the more restrictive of the two, and the one stated by the
dataset's own authors on their own repo) as the operative license until/unless clarified.

SOURCE CHANGE (2026-09-16): this originally downloaded seven per-country zips from Sekilab's own S3
bucket (bigdatacup.s3.ap-northeast-1.amazonaws.com/.../Country_Specific_Data_CRDDC2022/...). That
bucket now returns HTTP 403 Forbidden on every file, confirmed independently from two unrelated
networks - the bucket's access policy appears to have changed, not a transient fluke. This version
instead downloads FigShare's official combined mirror of the same dataset (one zip, all six
countries, published by the RDD2022/CRDDC2022 organizers themselves), verified by MD5 checksum
against FigShare's own published hash so a partial/corrupted 12GB+ download is caught rather than
silently producing bad data. If FigShare's link ever breaks too, check
https://github.com/sekilab/RoadDamageDetector for current mirror links before assuming this script
is broken.

Because this is one combined zip (not one zip per country), there's no way to download only a
subset of countries - the whole ~12.35GB has to come down regardless. --countries now only controls
which countries get linked into data/raw/ for the conversion step afterward (convert_voc_to_yolo.py
reads data/raw/<country>/), not what gets downloaded.

The zip's exact internal folder layout was not independently verified before writing this script
(Sekilab's own "Directory_Structure_CRDDC_RDD2022.txt" reference file was unreachable from this
environment - see reorganize_countries() below for how this script copes with that uncertainty at
extraction time instead of assuming a fixed layout).

NOTE (2026-09-17, Kaggle handoff): on the original Mac/home-network run, aria2c's multi-connection
mode was confirmed to get an immediate HTTP 403 from FigShare's CDN regardless of connection count
(1, 4, or 16) - the CDN appears to reject Range/segmented requests outright, not just high
concurrency. So on Kaggle this will (correctly) fall back to the plain single-connection urllib
path every time; that's expected, not a bug - the win here is Kaggle's raw single-connection
bandwidth to this host, not multi-connection parallelism.

NOTE (2026-09-17, Kaggle dataset cache): every phase that needs a fresh Kaggle kernel re-downloads
this same ~12.35GB zip from scratch, since /kaggle/working doesn't persist across kernel sessions -
phase 2 confirmed this the hard way. To stop paying that cost every phase, a one-time "cache
builder" kernel downloads + MD5-verifies the zip once and its output becomes a private Kaggle
Dataset; every later notebook just adds that Dataset as an input.

UPDATE (2026-09-17, same day): the cache Dataset did NOT turn out to contain a re-downloadable zip.
Kaggle's "New Dataset from notebook output" flow recursively auto-extracts every .zip file it finds
in the output - not just the outer combined zip, but the 7 per-country zips nested inside it too -
so the Dataset actually ended up holding a fully-extracted, ready-to-convert per-country directory
tree (confirmed by browsing the Dataset's Data Explorer: data/zips/RDD2022_released_through_CRDDC2022/
RDD2022/<Country>/<Country>/{train,test}/... for all 7 countries, ~85.8k files, 13.83GB). That's
actually a BETTER cache than a zip would have been - it skips the extraction step too, not just the
download - so find_kaggle_cached_extracted_countries() below is checked FIRST and, when it covers
every requested country, this script skips straight to "done" (no download, no MD5 verify, nothing -
extract_convert_per_country.py finds and uses the same cache independently). find_kaggle_cached_zip()
is kept as a fallback for a cache Dataset built some other way (e.g. zip auto-extraction turned off),
and the plain FigShare download remains the final fallback - so this script still works unchanged
off-Kaggle, or on a fresh Kaggle account with no cache Dataset set up yet.

Usage:
    python scripts/download_rdd2022.py                        # download + extract + link all found countries
    python scripts/download_rdd2022.py --countries Japan Czech # only link these two after extraction
    python scripts/download_rdd2022.py --skip-extract          # download (+ verify) only
    python scripts/download_rdd2022.py --connections 32        # more aria2c connections (default 16)
"""
import argparse
import hashlib
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

FIGSHARE_URL = "https://ndownloader.figshare.com/files/38030910"
FIGSHARE_ZIP_NAME = "RDD2022_released_through_CRDDC2022.zip"
FIGSHARE_MD5 = "b62bd51d2ffcfaa76c60f234f0cc2bb3"

# Logical country name -> name(s) we'll look for (case-insensitively) among directories inside the
# extracted zip, since the exact internal layout wasn't independently verified (see module docstring).
COUNTRY_ALIASES = {
    "Japan": ["Japan"],
    "India": ["India"],
    "Czech": ["Czech"],
    "Norway": ["Norway"],
    "United_States": ["United_States", "United States", "US", "USA"],
    "China_MotorBike": ["China_MotorBike", "China-MotorBike", "China_Motorbike"],
    "China_Drone": ["China_Drone", "China-Drone"],
}

DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"


def _format_bytes(n):
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}TB"


def download_with_aria2(url, dest_path, connections=16, retries=3):
    """Segmented, multi-connection download via the aria2c CLI - typically several times faster than
    a single urllib stream on hosts like FigShare's S3-backed CDN, where per-connection throughput
    is often capped well below the link's actual bandwidth. aria2c handles its own retries/resume
    (via its .aria2 control file next to the output), so this just shells out and lets it manage
    that; --continue=true means re-running after an interrupted aria2c download resumes rather than
    restarting from zero.
    """
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "aria2c",
        "-x", str(connections),          # max connections per server
        "-s", str(connections),          # split file into this many pieces
        "-k", "1M",                      # min split size
        "--continue=true",
        "--max-tries", str(retries),
        "--retry-wait=5",
        "--file-allocation=none",
        "--summary-interval=5",
        # FigShare's CDN (via ndownloader.figshare.com -> a presigned S3 URL) returns 403 to aria2c's
        # default "aria2/x.y.z" User-Agent - matching the header the plain-urllib path already used
        # successfully fixes it. --auto-file-renaming=false avoids aria2 silently writing to a
        # "-1" suffixed file if dest_path.name already exists from an earlier failed attempt.
        "--user-agent=Mozilla/5.0",
        "--auto-file-renaming=false",
        "-d", str(dest_path.parent),
        "-o", dest_path.name,
        url,
    ]
    print(f"  using aria2c with {connections} parallel connections (much faster than a single stream)")
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f"aria2c exited with code {result.returncode} - see its output above for details")


def download_one(url, dest_path, retries=3, connections=16):
    """Download url to dest_path. Skips entirely if dest_path already exists and is non-empty - this
    does NOT check the existing file's checksum, so a corrupted prior download won't be caught here;
    verify_md5() below is what actually guarantees integrity, run separately after this.

    Uses aria2c (multi-connection, much faster) when it's installed on PATH; otherwise falls back to
    a plain single-connection urllib download and prints a one-time hint about installing aria2c for
    a large file like this one. `brew install aria2` on macOS. If aria2c is present but fails (a
    misbehaving CDN, a network that blocks it, etc.), falls back to the urllib path automatically
    rather than giving up outright - not verified against every possible aria2c/network combination,
    so this fallback matters.
    """
    if dest_path.exists() and dest_path.stat().st_size > 0:
        print(f"  already have {dest_path.name} ({_format_bytes(dest_path.stat().st_size)}), skipping download")
        return

    dest_path.parent.mkdir(parents=True, exist_ok=True)

    if shutil.which("aria2c"):
        try:
            download_with_aria2(url, dest_path, connections=connections, retries=retries)
            return
        except RuntimeError as exc:
            print(f"  aria2c failed ({exc}); falling back to a plain single-connection download instead.")
            # clean up whatever partial/zero-byte file aria2c may have left behind before falling back
            if dest_path.exists() and dest_path.stat().st_size == 0:
                dest_path.unlink()
    else:
        print("  NOTE: aria2c not found on PATH - using a single-connection download, which will be "
              "noticeably slower for a file this size. `brew install aria2` (macOS) and re-run for a "
              "multi-connection download instead.")

    _download_urllib(url, dest_path, retries=retries)


def find_kaggle_cached_zip():
    """Look for a pre-cached copy of the combined zip under /kaggle/input/ - a private Kaggle
    Dataset added as this notebook's input, built once by a "cache builder" kernel that just runs
    `download_rdd2022.py --skip-extract` and turns its output into a Dataset (see the module
    docstring's 2026-09-17 note). Kaggle mounts every added dataset read-only at
    /kaggle/input/<dataset-slug>/, so this searches by filename across ALL mounted datasets rather
    than assuming a specific slug - the cache dataset can be renamed, or other unrelated datasets
    can be mounted alongside it, without breaking this. Returns None (not an error) when
    /kaggle/input doesn't exist at all (i.e. not running on Kaggle) or no dataset has the file.
    """
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.is_dir():
        return None
    matches = list(kaggle_input.rglob(FIGSHARE_ZIP_NAME))
    if not matches:
        return None
    if len(matches) > 1:
        print(f"  NOTE: found {len(matches)} copies of {FIGSHARE_ZIP_NAME} under /kaggle/input/, "
              f"using the first: {matches[0]}")
    return matches[0]


def find_kaggle_cached_extracted_countries(root=None):
    """Look for a pre-EXTRACTED per-country RDD2022 tree under /kaggle/input/ - what the cache
    Dataset actually contains (see the module docstring's 2026-09-17 UPDATE). Kaggle's "New Dataset
    from notebook output" recursively auto-unzips every .zip it finds in the output, including zips
    nested inside other zips - so the cache-builder notebook's output (which only ever contained the
    still-zipped combined archive on disk) turned into a fully-extracted directory tree by the time
    it became a Dataset: both the outer combined zip AND all 7 nested per-country zips got expanded
    in place, not just the outer one. That's a BETTER cache than a re-downloadable zip (skips
    extraction too, not just download), so this is checked before find_kaggle_cached_zip() above.

    Returns {country: path} for every COUNTRY_ALIASES key found as a non-empty directory somewhere
    under root (default /kaggle/input). Does ONE pass over the whole mounted-input tree rather than
    one rglob per country - /kaggle/input can hold 80k+ files once a dataset like this is extracted,
    so scanning it 7x over would be wasteful. Returns {} (not an error) when root doesn't exist (e.g.
    off-Kaggle) or nothing matches. `root` is overridable for self-testing without touching the real
    /kaggle/input.
    """
    kaggle_input = Path(root) if root is not None else Path("/kaggle/input")
    if not kaggle_input.is_dir():
        return {}
    found = {}
    for path in kaggle_input.rglob("*"):
        if len(found) == len(COUNTRY_ALIASES):
            break
        if path.name not in COUNTRY_ALIASES or path.name in found:
            continue
        if not path.is_dir():
            continue
        try:
            if any(path.iterdir()):
                found[path.name] = path
        except OSError:
            continue
    return found


def _download_urllib(url, dest_path, retries=3):
    """Plain single-connection streaming download - the fallback used when aria2c isn't available or
    didn't work."""
    tmp_path = dest_path.with_suffix(dest_path.suffix + ".part")

    for attempt in range(1, retries + 1):
        try:
            print(f"  downloading {url} -> {dest_path} (attempt {attempt}/{retries})")
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as resp, open(tmp_path, "wb") as out:
                total = int(resp.headers.get("Content-Length", 0))
                downloaded = 0
                chunk_size = 1024 * 1024
                last_print = time.time()
                while True:
                    chunk = resp.read(chunk_size)
                    if not chunk:
                        break
                    out.write(chunk)
                    downloaded += len(chunk)
                    if time.time() - last_print > 2:
                        pct = f"{downloaded / total:.0%}" if total else "?"
                        print(f"    {_format_bytes(downloaded)}" + (f" / {_format_bytes(total)} ({pct})" if total else ""),
                              end="\r", file=sys.stderr)
                        last_print = time.time()
            tmp_path.rename(dest_path)
            print(f"\n  done: {dest_path.name} ({_format_bytes(dest_path.stat().st_size)})")
            return
        except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, ConnectionError) as exc:
            print(f"\n  attempt {attempt} failed: {exc}")
            if tmp_path.exists():
                tmp_path.unlink()
            if attempt == retries:
                raise
            time.sleep(3 * attempt)


def verify_md5(path, expected_md5):
    """Stream-hash path and compare against expected_md5. Caches a good result next to the file
    (a .md5ok marker) so re-running this script doesn't re-hash a ~12GB file every time - if you
    ever suspect the downloaded file got corrupted after the fact, delete the .md5ok marker (or the
    zip itself) to force a real re-check.
    """
    marker = path.with_suffix(path.suffix + ".md5ok")
    if marker.exists():
        print(f"  {path.name}: MD5 previously verified (delete {marker.name} to re-check)")
        return True

    print(f"  verifying MD5 of {path.name} ({_format_bytes(path.stat().st_size)}, this can take a minute)...")
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(8 * 1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    actual = h.hexdigest()
    if actual != expected_md5:
        print(f"  MD5 MISMATCH: expected {expected_md5}, got {actual}")
        print(f"  {path} is corrupt or incomplete - delete it and re-run this script to re-download.")
        return False
    marker.write_text("ok\n")
    print(f"  MD5 verified: {actual}")
    return True


def extract_one(zip_path, extract_root):
    """Extract zip_path into extract_root, skipping if already extracted (marker-based, not by
    checking for a specific internal folder name - see reorganize_countries() for why)."""
    marker = extract_root / ".extracted"
    if marker.exists():
        print(f"  already extracted into {extract_root}, skipping")
        return
    extract_root.mkdir(parents=True, exist_ok=True)
    print(f"  extracting {zip_path.name} -> {extract_root} (large archive, this can take several minutes)")
    with zipfile.ZipFile(zip_path) as zf:
        bad = zf.testzip()
        if bad is not None:
            raise RuntimeError(f"{zip_path} is corrupt (bad member: {bad}) - delete it and re-run to re-download")
        zf.extractall(extract_root)
    marker.write_text("ok\n")


def reorganize_countries(extract_root, raw_dir, wanted_countries):
    """Find each wanted country's directory somewhere inside the extracted tree and link it to
    raw_dir/<country>, so convert_voc_to_yolo.py (which expects data/raw/<country>/ folders) keeps
    working unchanged regardless of whatever top-level wrapper folder(s) FigShare's zip actually
    uses internally.

    Uses a symlink rather than copying, since the extracted tree is already ~12GB+ and duplicating
    it serves no purpose. For each country, searches for directories whose name matches one of its
    known aliases (case-insensitive) and picks the SHALLOWEST match; if more than one directory at
    that same shallowest depth matches (e.g. the zip ships both a train/ and test/ split each with
    their own per-country subfolder), this prints every candidate found and picks the first
    alphabetically - flagged clearly so you can sanity-check it, rather than silently guessing.
    This is exactly the "raw data doesn't match documentation" messiness this project is meant to
    surface, so warn rather than hide it.
    """
    raw_dir.mkdir(parents=True, exist_ok=True)
    found_any = False

    for country in wanted_countries:
        link_path = raw_dir / country
        if link_path.exists() or link_path.is_symlink():
            print(f"  {country}: {link_path} already exists, leaving as-is")
            found_any = True
            continue

        aliases_lower = {a.lower() for a in COUNTRY_ALIASES[country]}
        candidates = [
            p for p in extract_root.rglob("*")
            if p.is_dir() and p.name.lower() in aliases_lower
        ]
        if not candidates:
            print(f"  WARNING: no directory matching {COUNTRY_ALIASES[country]} found under {extract_root} "
                  f"- {country} will be missing from data/raw/. Check the extracted tree by hand "
                  f"(e.g. `find {extract_root} -iname '*{country.split('_')[0]}*' -type d`) and symlink "
                  f"it manually if this script guessed wrong.")
            continue

        min_depth = min(len(p.relative_to(extract_root).parts) for p in candidates)
        shallowest = sorted(p for p in candidates if len(p.relative_to(extract_root).parts) == min_depth)
        if len(shallowest) > 1:
            print(f"  NOTE: multiple equally-shallow matches for {country}, picking the first:")
            for p in shallowest:
                print(f"    - {p.relative_to(extract_root)}")
        chosen = shallowest[0]
        link_path.symlink_to(chosen, target_is_directory=True)
        print(f"  {country}: linked data/raw/{country} -> {chosen.relative_to(extract_root)}")
        found_any = True

    if not found_any:
        print("  WARNING: none of the requested countries were found - inspect the extracted tree "
              f"under {extract_root} directly; the assumed alias names in COUNTRY_ALIASES may not "
              "match this zip's actual layout.")


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR),
                         help="root data directory (default: %(default)s)")
    parser.add_argument("--countries", nargs="+", choices=list(COUNTRY_ALIASES) + ["China"], default=None,
                         help="subset of countries to link into data/raw/ after extraction (default: all). "
                              "Does NOT reduce download size - FigShare ships one combined zip for all "
                              "six countries. Pass 'China' for both China_MotorBike and China_Drone.")
    parser.add_argument("--skip-extract", action="store_true", help="download (+ verify) only, don't unzip")
    parser.add_argument("--connections", type=int, default=16,
                         help="parallel connections for aria2c downloads (default: %(default)s, ignored "
                              "if aria2c isn't installed)")
    args = parser.parse_args(argv)

    countries = args.countries
    if countries is None:
        countries = list(COUNTRY_ALIASES)
    elif "China" in countries:
        countries = [c for c in countries if c != "China"] + ["China_MotorBike", "China_Drone"]

    cached_extracted = find_kaggle_cached_extracted_countries()
    missing_from_extracted_cache = [c for c in countries if c not in cached_extracted]
    if cached_extracted and not missing_from_extracted_cache:
        print(f"Found all {len(countries)} requested countries already pre-extracted under "
              f"/kaggle/input/ (see the module docstring's 2026-09-17 UPDATE) - skipping the download "
              f"entirely, no zip needed at all:")
        for country in countries:
            print(f"  {country}: {cached_extracted[country]}")
        print("\nDone (nothing downloaded/extracted here). Next: python scripts/extract_convert_per_country.py "
              "will independently find and use this same cache.")
        return 0
    elif cached_extracted:
        print(f"NOTE: found a partial pre-extracted cache under /kaggle/input/ ({sorted(cached_extracted)}) "
              f"but it's missing {missing_from_extracted_cache} - falling back to the normal "
              f"download/zip-cache path below for all requested countries (not mixing sources).")

    data_dir = Path(args.data_dir)
    zips_dir = data_dir / "zips"
    extract_root = data_dir / "raw" / "_extracted_all"
    raw_dir = data_dir / "raw"

    zip_path = zips_dir / FIGSHARE_ZIP_NAME
    if zip_path.exists() and zip_path.stat().st_size > 0:
        print(f"Already have {zip_path} ({_format_bytes(zip_path.stat().st_size)}), skipping download/cache-copy")
    else:
        cached = find_kaggle_cached_zip()
        if cached:
            print(f"Found a pre-cached copy at {cached} (Kaggle input dataset) - copying into "
                  f"{zip_path} instead of downloading ~12.35GB from FigShare again")
            zips_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(cached, zip_path)
            print(f"  copied {_format_bytes(zip_path.stat().st_size)}")
        else:
            print(f"Downloading combined RDD2022 archive (~12.35GB) from FigShare into {zip_path}")
            download_one(FIGSHARE_URL, zip_path, connections=args.connections)

    if not verify_md5(zip_path, FIGSHARE_MD5):
        print("\nAborting - downloaded file failed MD5 verification. Delete the zip and re-run.")
        return 1

    if args.skip_extract:
        print("\n--skip-extract set, stopping after download+verify.")
        return 0

    print(f"\nExtracting into {extract_root}")
    extract_one(zip_path, extract_root)

    print(f"\nLinking {len(countries)} countries into {raw_dir}")
    reorganize_countries(extract_root, raw_dir, countries)

    print("\nDone. Raw data is under:", raw_dir)
    print("Next: python scripts/convert_voc_to_yolo.py")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/convert_voc_to_yolo.py
"""Convert RDD2022's PASCAL-VOC-XML annotations into YOLO format, and build a unified manifest
across all six countries (roadmap: infra-defect-detection, phase 1).

Why this exists rather than just pointing Ultralytics at the raw VOC XML: (1) YOLO training wants
one .txt label per image with normalized [class cx cy w h] rows, not VOC's per-object XML; (2) more
importantly for this project, we need a single manifest that records which COUNTRY every image
came from, because phase 3's whole point is training on a subset of countries and evaluating
cross-country generalization - that experiment is impossible without country labels surviving the
conversion step.

Design choice - discover files by globbing + matching by filename stem, not by assuming a fixed
"images/" + "annotations/xmls/" subfolder layout: RDD2022's own directory-structure reference file
was unreachable when this project was set up (see download_rdd2022.py's docstring), so hardcoding
an assumed layout would risk silently processing zero files if the real layout differs. Globbing
recursively is slower but correct regardless of how each country's zip is actually organized
internally.

Damage classes (from the RDD2022/CRDDC'2022 label map):
    D00 - longitudinal crack
    D10 - transverse crack
    D20 - alligator crack
    D40 - pothole
Any other class name encountered (older RDD releases had more, e.g. D01/D11/D43/D44/D50) is logged
and SKIPPED, not silently merged into the nearest class - if you see a nontrivial skip count for a
country, that's worth a manual look before assuming the data converted cleanly.

Usage:
    python scripts/convert_voc_to_yolo.py                  # converts every country under data/raw/
    python scripts/convert_voc_to_yolo.py --countries Japan Czech
"""
import argparse
import csv
import sys
import xml.etree.ElementTree as ET
from pathlib import Path

try:
    from PIL import Image
except ImportError:
    Image = None

DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"

CLASS_MAP = {"D00": 0, "D10": 1, "D20": 2, "D40": 3}
CLASS_NAMES = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole"]


def find_files(root, suffix):
    return sorted(p for p in root.rglob(f"*{suffix}") if p.is_file())


def parse_voc_xml(xml_path):
    """Return (image_filename, width, height, [(class_name, xmin, ymin, xmax, ymax), ...]).

    width/height come from the XML's <size> block when present; if absent or zero (seen in some
    messy real-world VOC exports), the caller falls back to opening the image with PIL - this is
    exactly the kind of "annotation doesn't quite match what a clean benchmark would give you"
    messiness the project is supposed to be diagnosing, so we handle it rather than crash on it.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename_el = root.find("filename")
    image_filename = filename_el.text.strip() if filename_el is not None and filename_el.text else xml_path.stem + ".jpg"

    size_el = root.find("size")
    width = height = 0
    if size_el is not None:
        w_el, h_el = size_el.find("width"), size_el.find("height")
        width = int(w_el.text) if w_el is not None and w_el.text else 0
        height = int(h_el.text) if h_el is not None and h_el.text else 0

    objects = []
    for obj in root.findall("object"):
        name_el = obj.find("name")
        bnd = obj.find("bndbox")
        if name_el is None or bnd is None:
            continue
        name = (name_el.text or "").strip()
        try:
            xmin = float(bnd.find("xmin").text)
            ymin = float(bnd.find("ymin").text)
            xmax = float(bnd.find("xmax").text)
            ymax = float(bnd.find("ymax").text)
        except (AttributeError, TypeError, ValueError):
            continue
        objects.append((name, xmin, ymin, xmax, ymax))

    return image_filename, width, height, objects


def voc_box_to_yolo_line(class_idx, xmin, ymin, xmax, ymax, width, height):
    cx = (xmin + xmax) / 2 / width
    cy = (ymin + ymax) / 2 / height
    w = (xmax - xmin) / width
    h = (ymax - ymin) / height
    return f"{class_idx} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def convert_country(country, raw_dir, processed_dir, manifest_rows, stats):
    xml_files = find_files(raw_dir, ".xml")
    jpg_files = find_files(raw_dir, ".jpg") + find_files(raw_dir, ".jpeg") + find_files(raw_dir, ".JPG")
    image_by_stem = {}
    for p in jpg_files:
        image_by_stem.setdefault(p.stem, p)

    if not xml_files:
        print(f"  WARNING: no .xml annotation files found under {raw_dir} - is the zip actually extracted here?")
        return

    out_images = processed_dir / country / "images"
    out_labels = processed_dir / country / "labels"
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    n_ok = n_orphan_xml = n_bad_size = n_no_objects = n_skipped_class = 0

    for xml_path in xml_files:
        image_filename, width, height, objects = parse_voc_xml(xml_path)
        stem = Path(image_filename).stem

        image_path = image_by_stem.get(stem) or image_by_stem.get(xml_path.stem)
        if image_path is None:
            n_orphan_xml += 1
            continue

        if width <= 0 or height <= 0:
            if Image is None:
                n_bad_size += 1
                continue
            try:
                with Image.open(image_path) as im:
                    width, height = im.size
            except Exception:
                n_bad_size += 1
                continue

        yolo_lines = []
        for name, xmin, ymin, xmax, ymax in objects:
            if name not in CLASS_MAP:
                n_skipped_class += 1
                stats["skipped_classes"][name] = stats["skipped_classes"].get(name, 0) + 1
                continue
            xmin, xmax = sorted((max(0, xmin), min(width, xmax)))
            ymin, ymax = sorted((max(0, ymin), min(height, ymax)))
            if xmax <= xmin or ymax <= ymin:
                continue
            yolo_lines.append(voc_box_to_yolo_line(CLASS_MAP[name], xmin, ymin, xmax, ymax, width, height))
            stats["class_counts"][name] = stats["class_counts"].get(name, 0) + 1

        if not yolo_lines:
            n_no_objects += 1
            continue

        dest_stem = f"{country}__{stem}"
        dest_image = out_images / f"{dest_stem}{image_path.suffix.lower()}"
        dest_label = out_labels / f"{dest_stem}.txt"
        if not dest_image.exists():
            dest_image.write_bytes(image_path.read_bytes())
        dest_label.write_text("\n".join(yolo_lines) + "\n")

        manifest_rows.append({
            "country": country,
            "image": str(dest_image.relative_to(processed_dir.parent)),
            "label": str(dest_label.relative_to(processed_dir.parent)),
            "width": width,
            "height": height,
            "num_objects": len(yolo_lines),
            "classes": ";".join(sorted({l.split()[0] for l in yolo_lines})),
        })
        n_ok += 1

    print(f"  {country}: {n_ok} converted, {n_orphan_xml} orphan xml (no matching image), "
          f"{n_bad_size} bad/unreadable size, {n_no_objects} had zero valid objects after class "
          f"filtering, {n_skipped_class} individual boxes skipped for unrecognized class names")
    stats["per_country"][country] = {
        "converted": n_ok, "orphan_xml": n_orphan_xml, "bad_size": n_bad_size,
        "no_objects": n_no_objects, "skipped_class_boxes": n_skipped_class,
    }


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR))
    parser.add_argument("--countries", nargs="+", default=None,
                         help="subset of country folder names under data/raw/ to convert (default: all found)")
    args = parser.parse_args(argv)

    if Image is None:
        print("NOTE: Pillow not installed - images with missing/zero <size> in their XML will be "
              "skipped instead of measured. `pip install Pillow` to handle those too.", file=sys.stderr)

    data_dir = Path(args.data_dir)
    raw_dir = data_dir / "raw"
    processed_dir = data_dir / "processed"

    if args.countries:
        countries = args.countries
    else:
        countries = sorted(p.name for p in raw_dir.iterdir() if p.is_dir())

    if not countries:
        print(f"No country folders found under {raw_dir} - run download_rdd2022.py first.", file=sys.stderr)
        return 1

    manifest_rows = []
    stats = {"class_counts": {}, "skipped_classes": {}, "per_country": {}}

    print(f"Converting {len(countries)} countries: {countries}")
    for country in countries:
        print(f"\n[{country}]")
        convert_country(country, raw_dir / country, processed_dir, manifest_rows, stats)

    manifest_path = data_dir / "manifest.csv"
    with open(manifest_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["country", "image", "label", "width", "height", "num_objects", "classes"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    dataset_yaml = data_dir / "dataset.yaml"
    dataset_yaml.write_text(
        "# Auto-generated by convert_voc_to_yolo.py - combined view across all converted countries.\n"
        "# For the phase-3 cross-country experiments, build per-experiment yaml files that point at\n"
        "# a subset of countries' image folders instead of reusing this combined one directly.\n"
        f"path: {processed_dir}\n"
        "train: */images\n"
        f"nc: {len(CLASS_NAMES)}\n"
        f"names: {CLASS_NAMES}\n"
    )

    print(f"\nWrote manifest ({len(manifest_rows)} images) to {manifest_path}")
    print(f"Wrote combined dataset.yaml to {dataset_yaml}")
    print("\nClass distribution across all converted countries:")
    for name, idx in CLASS_MAP.items():
        print(f"  {name}: {stats['class_counts'].get(name, 0)}")
    if stats["skipped_classes"]:
        print("\nSkipped (unrecognized) class names encountered - investigate before trusting counts above:")
        for name, count in sorted(stats["skipped_classes"].items(), key=lambda kv: -kv[1]):
            print(f"  {name!r}: {count}")

    print("\nNext: python scripts/eda_report.py")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/extract_convert_per_country.py
"""Extract + convert RDD2022 ONE COUNTRY AT A TIME, deleting each country's intermediate data as
soon as it's converted (roadmap: infra-defect-detection, phase 1 - Kaggle disk-constrained variant).

WHY THIS EXISTS (2026-09-17, v2->v3->v4): the straightforward approach - download_rdd2022.py's normal
extract_one()/reorganize_countries(), which calls zf.extractall() on the WHOLE combined zip at once
- needs the 12.35GB combined zip AND all the extracted raw VOC data on disk simultaneously, which
blew through a Kaggle notebook's working-directory quota on the first attempt.

v2 tried to fix this by extracting one country's files at a time straight out of the combined zip's
member list - but that assumed the combined zip was a flat tree of images/XML per country. It
ISN'T: FigShare's combined zip (b62bd51d2ffcfaa76c60f234f0cc2bb3, the officially-published MD5) is
actually a ZIP-OF-ZIPS - exactly 7 entries, one per country, each itself a complete nested .zip
(confirmed 2026-09-17 by actually listing the real zip's namelist on Kaggle: `RDD2022/Japan.zip`,
`RDD2022/India.zip`, `RDD2022/Czech.zip`, `RDD2022/Norway.zip`, `RDD2022/United_States.zip`,
`RDD2022/China_MotorBike.zip`, `RDD2022/China_Drone.zip`). v2's matching logic looked for a path
COMPONENT exactly equal to a country alias (e.g. a directory literally named "Japan"), which never
matched because the actual component is "Japan.zip" (a file, not a directory) - so v2 silently
converted 0 images across all 7 countries and wrote an empty manifest.csv.

v3 fixed the matching (match by filename STEM instead of path component) but still copied each
country's nested zip out to a temp file ON DISK before extracting it, while the 12.35GB combined zip
stayed on disk the whole time. That's fine for the small countries, but Norway's nested zip alone is
9.9GB - so by the time v3 reached Norway, disk needed 12.35GB (combined zip, still present, only
deleted at the very end) + already-converted data from the 5 prior countries + a 9.9GB temp copy of
Norway's nested zip, all at once. That blew past Kaggle's 19.5GiB /kaggle/working quota with
`OSError: [Errno 28] No space left on device` mid-copy - not a transient hiccup, this was guaranteed
to happen as soon as processing reached the largest country, regardless of retry.

v4 fixes the actual disk-budget problem instead of just the matching bug:
  1. Read EVERY country's nested-zip bytes into RAM first (`ZipFile.read()`, not `copyfileobj` to a
     temp file) - Kaggle's RAM (~31GB, most of it free) isn't quota-limited the way /kaggle/working
     is, so holding all 7 countries' compressed bytes (summing to the same ~12.35GB as the combined
     zip) in memory costs nothing against the disk quota.
  2. Only ONCE ALL SEVEN have been read into memory - and the combined zip is no longer needed for
     anything - close it and delete the 12.35GB file from disk, BEFORE extracting a single country to
     disk. This frees the full 19.5GiB quota (minus whatever's already used) for the extraction step,
     regardless of which country happens to be biggest or what order they're processed in.
  3. Per country: extract from an in-memory `io.BytesIO` (no on-disk temp zip at all), convert, delete
     the extracted raw folder, and drop that country's bytes from the in-memory dict - so RAM usage
     shrinks back down as we go instead of holding all 7 for the whole run.

v5 (2026-09-17, same day as v4): check for a pre-EXTRACTED per-country cache under /kaggle/input/
FIRST, before any of the v4 zip-reading machinery - see download_rdd2022.py's module docstring
2026-09-17 UPDATE for why the Kaggle Dataset cache ended up holding fully-extracted directories
rather than a re-usable zip (Kaggle auto-extracts zips-within-zips when building a Dataset from
notebook output). When the cache covers every requested country this skips the ENTIRE v4 dance -
convert_country() only ever reads from raw_dir, never writes into it, so it's safe to point it
straight at the read-only /kaggle/input cache path with no copy at all. See
_write_manifest_and_summary() and main() below.

Usage:
    python scripts/extract_convert_per_country.py                  # all countries, then deletes the combined zip
    python scripts/extract_convert_per_country.py --keep-zip       # don't delete the combined zip early or at the end
                                                                    # (only use this if you have >32GB of quota - it
                                                                    # defeats the whole point of the v4 fix above)
    python scripts/extract_convert_per_country.py --data-dir data
"""
import argparse
import csv
import io
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import download_rdd2022 as dl          # noqa: E402  (FIGSHARE_ZIP_NAME, COUNTRY_ALIASES)
import convert_voc_to_yolo as cv       # noqa: E402  (convert_country, CLASS_MAP, CLASS_NAMES)


def _format_bytes(n):
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024:
            return f"{n:.1f}{unit}"
        n /= 1024
    return f"{n:.1f}TB"


DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"


def _print_disk_usage(label, working_dir=None):
    result = subprocess.run(["df", "-h", "/"], capture_output=True, text=True)
    print(f"  [{label}] df -h /:\n" + "\n".join("    " + l for l in result.stdout.splitlines()))
    # df -h / reports the container's overall overlay filesystem, which on Kaggle does NOT move in
    # step with /kaggle/working - the thing actually counted against the 19.5GiB quota is disk usage
    # UNDER /kaggle/working specifically, which `du` reports correctly.
    if working_dir is not None:
        du = subprocess.run(["du", "-sh", str(working_dir)], capture_output=True, text=True)
        print(f"  [{label}] du -sh {working_dir}: {du.stdout.strip() or du.stderr.strip()}")


def _write_manifest_and_summary(data_dir, processed_dir, manifest_rows, stats):
    """Shared tail for both the pre-extracted-cache fast path and the full zip-reading slow path in
    main(): write manifest.csv + dataset.yaml and print the class-distribution summary. Factored out
    so the two paths can't silently drift apart on what "done" means."""
    manifest_path = data_dir / "manifest.csv"
    with open(manifest_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["country", "image", "label", "width", "height", "num_objects", "classes"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    dataset_yaml = data_dir / "dataset.yaml"
    dataset_yaml.write_text(
        "# Auto-generated by extract_convert_per_country.py - combined view across all converted countries.\n"
        "# For the phase-3 cross-country experiments, build per-experiment yaml files that point at\n"
        "# a subset of countries' image folders instead of reusing this combined one directly.\n"
        f"path: {processed_dir}\n"
        "train: */images\n"
        f"nc: {len(cv.CLASS_NAMES)}\n"
        f"names: {cv.CLASS_NAMES}\n"
    )

    print(f"\nWrote manifest ({len(manifest_rows)} images) to {manifest_path}")
    print(f"Wrote combined dataset.yaml to {dataset_yaml}")
    print("\nClass distribution across all converted countries:")
    for name in cv.CLASS_MAP:
        print(f"  {name}: {stats['class_counts'].get(name, 0)}")
    if stats["skipped_classes"]:
        print("\nSkipped (unrecognized) class names encountered - investigate before trusting counts above:")
        for name, count in sorted(stats["skipped_classes"].items(), key=lambda kv: -kv[1]):
            print(f"  {name!r}: {count}")

    if not manifest_rows:
        print("\nWARNING: manifest is EMPTY - 0 images were converted. Check the 'Entries matched' listing "
              "above for NOT FOUND countries or unmatched entries before trusting anything downstream.")


def match_country_by_stem(entry_name):
    """Which COUNTRY_ALIASES key this combined-zip entry is, by comparing its filename stem (the
    name with the LAST extension stripped, e.g. "RDD2022/China_MotorBike.zip" -> "China_MotorBike")
    against the known aliases, case-insensitively. This is the v3 fix: v2 matched whole path
    COMPONENTS looking for a directory named e.g. "Japan", which never matched because the real
    entries are files named "Japan.zip", not directories named "Japan"."""
    stem_lower = Path(entry_name).stem.lower()
    for country, aliases in dl.COUNTRY_ALIASES.items():
        for alias in aliases:
            if alias.lower() == stem_lower:
                return country
    return None


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR))
    parser.add_argument("--keep-zip", action="store_true",
                         help="don't delete the combined zip early (once all countries are read into "
                              "memory) or at the end - default is to delete it early, which is what "
                              "makes the largest country (Norway, ~9.9GB) fit under Kaggle's quota")
    parser.add_argument("--countries", nargs="+", default=None,
                         help="only extract+convert these countries (default: all 7). Added for phase 2, "
                              "which only needs a single country (e.g. --countries Czech) - still reads "
                              "the whole combined zip's member list, but only pulls the requested "
                              "countries' bytes into memory, so the early-delete-the-zip step still runs "
                              "and disk usage stays tiny.")
    args = parser.parse_args(argv)

    data_dir = Path(args.data_dir)
    zip_path = data_dir / "zips" / dl.FIGSHARE_ZIP_NAME
    raw_dir = data_dir / "raw"
    processed_dir = data_dir / "processed"
    working_dir = data_dir.resolve().parent  # e.g. /kaggle/working - what the quota actually tracks

    processed_dir.mkdir(parents=True, exist_ok=True)
    manifest_rows = []
    stats = {"class_counts": {}, "skipped_classes": {}, "per_country": {}}

    # v5 (2026-09-17, same day as v4 above): check for a pre-EXTRACTED per-country cache under
    # /kaggle/input/ first - see download_rdd2022.py's module docstring 2026-09-17 UPDATE for why the
    # cache Dataset ended up holding fully-extracted directories rather than a re-usable zip (Kaggle
    # auto-extracts zips-within-zips when building a Dataset from notebook output). When it covers
    # every requested country this skips the ENTIRE zip-reading/in-memory-extraction dance above -
    # convert_country() only ever reads from raw_dir (rglob for .xml/.jpg, then read_bytes()), never
    # writes into it, so it's safe to point it straight at the read-only /kaggle/input cache path with
    # no copy at all.
    requested_countries = args.countries if args.countries else list(dl.COUNTRY_ALIASES)
    cached_dirs = dl.find_kaggle_cached_extracted_countries()
    missing_from_cache = [c for c in requested_countries if c not in cached_dirs]

    if cached_dirs and not missing_from_cache:
        print(f"Found all {len(requested_countries)} requested countries pre-extracted under "
              f"/kaggle/input/ (a Kaggle Dataset cache) - converting straight from there, no zip "
              f"involved at all:")
        for country in requested_countries:
            print(f"  {country}: {cached_dirs[country]}")
        for country in requested_countries:
            print(f"\n[{country}] converting directly from the cached extracted tree...")
            cv.convert_country(country, cached_dirs[country], processed_dir, manifest_rows, stats)
        _write_manifest_and_summary(data_dir, processed_dir, manifest_rows, stats)
        print("\nNext: python scripts/eda_report.py")
        return 0
    elif cached_dirs:
        print(f"NOTE: found a partial pre-extracted cache under /kaggle/input/ ({sorted(cached_dirs)}) "
              f"but it's missing {missing_from_cache} - falling back to the full combined-zip path "
              f"below for ALL requested countries (not mixing sources, to keep this predictable).")

    if not zip_path.exists():
        print(f"{zip_path} not found - run download_rdd2022.py --skip-extract first.", file=sys.stderr)
        return 1

    # Clean slate for raw_dir: it's always transient intermediate storage, never a final deliverable,
    # so wipe any leftover partial state from a previous crashed run (e.g. a half-written
    # _nested_Norway.zip from the v3 run that hit the disk-quota error) before starting.
    if raw_dir.exists():
        print(f"Removing leftover {raw_dir} from a previous run (transient data only, safe to wipe)...")
        shutil.rmtree(raw_dir)
    raw_dir.mkdir(parents=True, exist_ok=True)

    print(f"Opening {zip_path} to read its member list (no extraction yet, costs no disk space)...")
    outer_zf = zipfile.ZipFile(zip_path)
    infos = [i for i in outer_zf.infolist() if not i.filename.endswith("/")]
    print(f"  combined zip contains {len(infos)} entries")

    by_country = {}
    unmatched = []
    for info in infos:
        country = match_country_by_stem(info.filename)
        if country:
            by_country[country] = info
        else:
            unmatched.append(info.filename)

    print("\nEntries matched (this is the combined zip's REAL layout - one nested zip per country):")
    for country in dl.COUNTRY_ALIASES:
        if country in by_country:
            info = by_country[country]
            print(f"  {country}: {info.filename} ({_format_bytes(info.file_size)})")
        else:
            print(f"  {country}: NOT FOUND")

    if args.countries:
        missing = [c for c in args.countries if c not in by_country]
        if missing:
            print(f"\n--countries requested {missing} but the combined zip doesn't have (a match for) "
                  f"{'them' if len(missing) > 1 else 'it'} - check spelling against COUNTRY_ALIASES.", file=sys.stderr)
            return 1
        by_country = {c: by_country[c] for c in args.countries}
        print(f"\n--countries set: only extracting {list(by_country.keys())} (out of all 7 found above)")

    if unmatched:
        print(f"\n  {len(unmatched)} entries matched no known country alias:")
        for n in unmatched:
            print(f"    {n}")

    _print_disk_usage("before reading anything", working_dir)

    # Step 1: read EVERY country's nested-zip bytes into RAM before extracting any of them to disk.
    # This is what lets us delete the 12.35GB combined zip BEFORE the largest country (Norway,
    # 9.9GB) needs to be extracted, instead of only being able to delete it at the very end.
    print(f"\nReading all {len(by_country)} countries' nested-zip bytes into memory (RAM isn't "
          f"quota-limited the way /kaggle/working is - this avoids ever needing the 12.35GB combined "
          f"zip and a country's extracted data on disk at the same time)...")
    country_bytes = {}
    for country, info in by_country.items():
        print(f"  reading {country} ({info.filename}, {_format_bytes(info.file_size)}) into memory...")
        country_bytes[country] = outer_zf.read(info)
    zip_size = zip_path.stat().st_size
    outer_zf.close()

    if not args.keep_zip:
        zip_path.unlink()
        print(f"\nDeleted {zip_path} ({_format_bytes(zip_size)}) early - every country's bytes are "
              f"already in memory, so the combined zip is no longer needed and this frees up disk "
              f"headroom before extracting any country.")
    else:
        print(f"\n--keep-zip set: leaving {zip_path} ({_format_bytes(zip_size)}) on disk (less headroom "
              f"for the extraction step below - only safe with a much larger quota than 19.5GiB).")
    _print_disk_usage("after reading all countries into memory" + ("" if args.keep_zip else " + deleting the combined zip"), working_dir)

    # Step 2: per country, extract from the in-memory bytes (no on-disk temp zip), convert, clean up.
    for country in list(country_bytes.keys()):
        data = country_bytes.pop(country)  # drop from the dict now so RAM shrinks as we go

        country_raw = raw_dir / country
        if country_raw.exists():
            shutil.rmtree(country_raw)
        country_raw.mkdir(parents=True)

        print(f"\n[{country}] extracting {_format_bytes(len(data))} (in memory) into {country_raw}...")
        with zipfile.ZipFile(io.BytesIO(data)) as inner_zf:
            bad = inner_zf.testzip()
            if bad is not None:
                print(f"  WARNING: {country}'s nested zip is corrupt (bad member: {bad}) - skipping {country}")
                shutil.rmtree(country_raw)
                del data
                continue
            inner_zf.extractall(country_raw)
        del data  # free this country's RAM now that it's on disk as extracted files

        print(f"  converting...")
        cv.convert_country(country, country_raw, processed_dir, manifest_rows, stats)

        print(f"  deleting {country_raw} (already converted, no longer needed) to free space for the next country...")
        shutil.rmtree(country_raw)
        _print_disk_usage(f"after {country}", working_dir)

    _write_manifest_and_summary(data_dir, processed_dir, manifest_rows, stats)
    _print_disk_usage("final", working_dir)
    print("\nNext: python scripts/eda_report.py")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/make_cross_country_split.py
"""Build the phase-3 cross-country split: merge several SOURCE countries into a train/val/
in-domain-test split, and write a separate full-image evaluation list for each TARGET country that
the model never trains on (roadmap: infra-defect-detection, phase 3 - distribution-shift diagnosis).

WHY THIS EXISTS: phase 2 answered "how good is a same-country baseline". Phase 3 asks "how much
worse does this get when the test country wasn't in training, and why" - that comparison needs two
things this script produces: (1) an IN-DOMAIN test split carved out of the source countries the same
way phase 2 did (held out, never trained on, reproducible via a recorded seed) as the "fair" number
to compare against, and (2) one full-image list per target country - no split, since the model never
sees any of a target country's images during training, so every one of its images is fair to
evaluate on.

Design (per the roadmap's phase-3 spec, chosen for a NATURAL - not synthetic - distribution shift):
  - source (train) countries: Japan, India, Czech - largest source-country image counts among the
    non-target countries per data/eda_report.md, plus Czech to keep phase 2's already-trained-on
    country in-domain rather than discarding that data.
  - target (cross-domain) countries: Norway, United_States, China_MotorBike, China_Drone - held out
    entirely from training so each can be reported as its own distribution-shift data point (motorbike-
    mounted vs. drone-mounted capture in China are different enough equipment/altitude conditions that
    lumping them into one "China" number would hide which one drives any gap).

Usage:
    python scripts/make_cross_country_split.py
    python scripts/make_cross_country_split.py --train-countries Japan India Czech \\
        --test-countries Norway United_States China_MotorBike China_Drone --seed 42
"""
import argparse
import csv
import json
import random
import sys
from collections import Counter
from pathlib import Path

DEFAULT_DATA_DIR = Path(__file__).resolve().parent.parent / "data"

CLASS_NAMES = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole"]

DEFAULT_TRAIN_COUNTRIES = ["Japan", "India", "Czech"]
DEFAULT_TEST_COUNTRIES = ["Norway", "United_States", "China_MotorBike", "China_Drone"]


def _write_image_list(list_path, rows, data_dir):
    with open(list_path, "w") as f:
        for r in rows:
            # manifest.csv's "image" column is relative to data_dir - resolve to an absolute path so
            # `yolo train`/`yolo val` behave the same regardless of the invoking directory.
            f.write(str((data_dir / r["image"]).resolve()) + "\n")


def _write_dataset_yaml(yaml_path, data_dir, comment, **splits):
    """splits: e.g. train=Path, val=Path, test=Path - only the keys given are written."""
    lines = [
        f"# {comment}\n",
        "# Labels are found by Ultralytics' convention: same path with the LAST 'images' path "
        "component\n# replaced by 'labels' and the extension replaced by .txt.\n",
    ]
    for key, path in splits.items():
        lines.append(f"{key}: {Path(path).resolve()}\n")
    lines.append(f"nc: {len(CLASS_NAMES)}\n")
    lines.append(f"names: {CLASS_NAMES}\n")
    yaml_path.write_text("".join(lines))


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--train-countries", nargs="+", default=DEFAULT_TRAIN_COUNTRIES,
                         help=f"countries merged into train/val/in-domain-test (default: {DEFAULT_TRAIN_COUNTRIES})")
    parser.add_argument("--test-countries", nargs="+", default=DEFAULT_TEST_COUNTRIES,
                         help=f"countries held out entirely, each getting its own full-image eval "
                              f"list (default: {DEFAULT_TEST_COUNTRIES})")
    parser.add_argument("--data-dir", default=str(DEFAULT_DATA_DIR))
    parser.add_argument("--train-frac", type=float, default=0.7)
    parser.add_argument("--val-frac", type=float, default=0.15)
    parser.add_argument("--seed", type=int, default=42, help="fixed shuffle seed - record this alongside any reported metric")
    parser.add_argument("--out-name", default="cross_country", help="subfolder under data/splits/ to write into")
    args = parser.parse_args(argv)

    overlap = set(args.train_countries) & set(args.test_countries)
    if overlap:
        print(f"--train-countries and --test-countries overlap: {sorted(overlap)} - a country must be "
              f"either fully in-domain (trained on) or fully held out, not both.", file=sys.stderr)
        return 1

    if args.train_frac + args.val_frac >= 1.0:
        print(f"--train-frac ({args.train_frac}) + --val-frac ({args.val_frac}) must leave room for a "
              f"non-empty in-domain-test split (currently sums to {args.train_frac + args.val_frac})",
              file=sys.stderr)
        return 1

    data_dir = Path(args.data_dir)
    manifest_path = data_dir / "manifest.csv"
    if not manifest_path.exists():
        print(f"{manifest_path} not found - run extract_convert_per_country.py first.", file=sys.stderr)
        return 1

    with open(manifest_path, newline="") as f:
        all_rows = list(csv.DictReader(f))

    source_rows = [r for r in all_rows if r["country"] in args.train_countries]
    missing_source = set(args.train_countries) - {r["country"] for r in source_rows}
    if missing_source:
        print(f"No rows for train-country(ies) {sorted(missing_source)} in {manifest_path} - check "
              f"spelling, or that extract_convert_per_country.py was run with these countries.",
              file=sys.stderr)
        return 1
    if not source_rows:
        print(f"No rows found for any of --train-countries {args.train_countries} in {manifest_path}.",
              file=sys.stderr)
        return 1

    target_rows_by_country = {c: [r for r in all_rows if r["country"] == c] for c in args.test_countries}
    missing_target = [c for c, rows in target_rows_by_country.items() if not rows]
    if missing_target:
        print(f"No rows for test-country(ies) {missing_target} in {manifest_path} - check spelling, "
              f"or that extract_convert_per_country.py was run with these countries.", file=sys.stderr)
        return 1

    # Sort first so the shuffle is deterministic regardless of manifest.csv's row order, then shuffle
    # ACROSS the merged source countries (not per-country) so train/val/in-domain-test each get a
    # representative mix rather than e.g. all of Japan in train and all of Czech in val by chance.
    source_rows.sort(key=lambda r: (r["country"], r["image"]))
    rng = random.Random(args.seed)
    rng.shuffle(source_rows)

    n = len(source_rows)
    n_train = int(n * args.train_frac)
    n_val = int(n * args.val_frac)
    splits = {
        "train": source_rows[:n_train],
        "val": source_rows[n_train:n_train + n_val],
        "in_domain_test": source_rows[n_train + n_val:],
    }

    split_dir = data_dir / "splits" / args.out_name
    split_dir.mkdir(parents=True, exist_ok=True)

    print(f"Source countries {args.train_countries}: {n} total images")
    per_split_country_counts = {}
    for split_name, split_rows in splits.items():
        list_path = split_dir / f"{split_name}.txt"
        _write_image_list(list_path, split_rows, data_dir)
        counts = dict(Counter(r["country"] for r in split_rows))
        per_split_country_counts[split_name] = counts
        print(f"  {split_name}: {len(split_rows)} images ({counts}) -> {list_path}")

    if any(len(v) == 0 for v in splits.values()):
        print(f"\nWARNING: at least one source split is empty - {args.train_countries} may not have "
              f"enough images for these fractions.", file=sys.stderr)

    dataset_yaml_path = split_dir / "dataset.yaml"
    _write_dataset_yaml(
        dataset_yaml_path, data_dir,
        comment=f"Auto-generated by make_cross_country_split.py. train_countries={args.train_countries}, seed={args.seed}.",
        train=split_dir / "train.txt",
        val=split_dir / "val.txt",
        test=split_dir / "in_domain_test.txt",
    )
    print(f"  dataset.yaml -> {dataset_yaml_path}")

    # Each target country: one full-image list (no split - the model never trains on any of it, so
    # every image is fair game for evaluation) and its own single-purpose eval yaml, so
    # train_cross_country_baseline.py can call model.val(data=<this yaml>, split="test") the SAME WAY
    # for the in-domain test set and for every target country - one code path, not a special case
    # per country.
    target_counts = {}
    for country, rows in target_rows_by_country.items():
        rows = sorted(rows, key=lambda r: r["image"])  # deterministic order, even though unsplit
        list_path = split_dir / f"eval_{country}.txt"
        _write_image_list(list_path, rows, data_dir)
        target_counts[country] = len(rows)
        eval_yaml_path = split_dir / f"eval_{country}.yaml"
        _write_dataset_yaml(
            eval_yaml_path, data_dir,
            comment=f"Auto-generated by make_cross_country_split.py. Full held-out eval set for "
                    f"target country={country} (never trained on) - use with "
                    f"model.val(data=this, split='test'). train/val below are NOT used for training "
                    f"(this yaml is only ever passed to model.val, never model.train) - they're set "
                    f"to the same list as 'test' only because Ultralytics' check_det_dataset() "
                    f"requires 'train' and 'val' keys to be present in every data yaml it loads.",
            train=list_path,
            val=list_path,
            test=list_path,
        )
        print(f"  target {country}: {len(rows)} images -> {list_path} (+ {eval_yaml_path.name})")

    split_config = {
        "train_countries": args.train_countries,
        "test_countries": args.test_countries,
        "seed": args.seed,
        "train_frac": args.train_frac,
        "val_frac": args.val_frac,
        "in_domain_test_frac": round(1.0 - args.train_frac - args.val_frac, 6),
        "n_source_total": n,
        "n_train": len(splits["train"]),
        "n_val": len(splits["val"]),
        "n_in_domain_test": len(splits["in_domain_test"]),
        "per_split_country_counts": per_split_country_counts,
        "target_country_counts": target_counts,
    }
    config_path = split_dir / "split_config.json"
    config_path.write_text(json.dumps(split_config, indent=2) + "\n")
    print(f"  split_config.json -> {config_path}")

    print(f"\nSource: {n} images -> train={len(splits['train'])} val={len(splits['val'])} "
          f"in_domain_test={len(splits['in_domain_test'])} (seed={args.seed})")
    print(f"Targets: {target_counts}")
    print(f"\nNext: python scripts/train_cross_country_baseline.py --data {dataset_yaml_path}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/train_cross_country_baseline.py
"""Train ONE YOLO11 model on merged source countries, then evaluate it separately on the in-domain
held-out test split AND every target (cross-domain) country, reporting the gap between them
(roadmap: infra-defect-detection, phase 3 - distribution-shift diagnosis).

WHY THIS EXISTS: phase 2 established a same-country baseline (train and test both Czech). Phase 3's
job is to quantify - not just intuit - how much worse detection gets when the test country wasn't in
training, and whether that gap is uniform or concentrated in specific classes/countries. That needs
exactly one trained model evaluated the SAME way (same code path, same metric definitions) on
multiple held-out sets: (1) the in-domain test split - drawn from the source countries but never
trained on, the "fair" number a cross-domain result should be compared against, since it isolates the
country-shift effect from ordinary train/test variance - and (2) each target country's full image
set, evaluated independently rather than pooled, so a report can say e.g. "Norway drops less than
China_Drone" instead of hiding that behind one averaged cross-domain number.

Usage:
    python scripts/train_cross_country_baseline.py --data data/splits/cross_country/dataset.yaml
    python scripts/train_cross_country_baseline.py --data data/splits/cross_country/dataset.yaml \\
        --model yolo11s.pt --epochs 100
"""
import argparse
import json
import sys
import time
from pathlib import Path

import yaml
from ultralytics import YOLO

DEFAULT_RUNS_DIR = Path(__file__).resolve().parent.parent / "runs" / "phase3"


def _class_names(data_cfg):
    return data_cfg.get("names", [])


def _summarize_box(box, class_names):
    per_class = []
    for idx in range(len(class_names)):
        if idx in box.ap_class_index:
            p, r, ap50, ap = box.class_result(list(box.ap_class_index).index(idx))
            per_class.append({"class": class_names[idx], "precision": float(p), "recall": float(r),
                               "ap50": float(ap50), "ap50_95": float(ap)})
        else:
            per_class.append({"class": class_names[idx], "precision": None, "recall": None,
                               "ap50": None, "ap50_95": None, "note": "no instances of this class in this eval set"})
    overall = {
        "precision_mean": float(box.mp),
        "recall_mean": float(box.mr),
        "map50": float(box.map50),
        "map50_95": float(box.map),
        "map75": float(box.map75),
    }
    return overall, per_class


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data", required=True, help="path to the cross-country dataset.yaml written by make_cross_country_split.py")
    parser.add_argument("--model", default="yolo11n.pt",
                         help="Ultralytics model/checkpoint to start from (default: yolo11n.pt - kept "
                              "the same as phase 2's baseline so the two are architecture-comparable; "
                              "phase 4 is where architecture itself becomes the variable).")
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--batch", type=int, default=16)
    parser.add_argument("--seed", type=int, default=42, help="must match (or be recorded alongside) the seed used for the split")
    parser.add_argument("--patience", type=int, default=20, help="early-stop patience (epochs with no val improvement)")
    parser.add_argument("--project", default=str(DEFAULT_RUNS_DIR))
    parser.add_argument("--name", default="cross_country_baseline", help="training run name")
    args = parser.parse_args(argv)

    data_yaml_path = Path(args.data)
    if not data_yaml_path.exists():
        print(f"{data_yaml_path} not found - run make_cross_country_split.py first.", file=sys.stderr)
        return 1

    with open(data_yaml_path) as f:
        data_cfg = yaml.safe_load(f)
    class_names = _class_names(data_cfg)

    split_dir = data_yaml_path.parent
    split_config_path = split_dir / "split_config.json"
    if not split_config_path.exists():
        print(f"{split_config_path} not found - {data_yaml_path} doesn't look like it was written by "
              f"make_cross_country_split.py (no split_config.json alongside it).", file=sys.stderr)
        return 1
    split_config = json.loads(split_config_path.read_text())
    test_countries = split_config["test_countries"]

    missing_eval_yamls = [c for c in test_countries if not (split_dir / f"eval_{c}.yaml").exists()]
    if missing_eval_yamls:
        print(f"Missing eval yaml(s) for {missing_eval_yamls} in {split_dir} - re-run "
              f"make_cross_country_split.py.", file=sys.stderr)
        return 1

    print(f"Training {args.model} on source countries {split_config['train_countries']} "
          f"(data={data_yaml_path}, seed={args.seed}, epochs={args.epochs}, imgsz={args.imgsz})...")
    print(f"  split (from {split_config_path.name}): train={split_config['n_train']} "
          f"val={split_config['n_val']} in_domain_test={split_config['n_in_domain_test']} "
          f"(split seed={split_config['seed']}, fractions "
          f"{split_config['train_frac']}/{split_config['val_frac']}/{split_config['in_domain_test_frac']})")
    print(f"  target countries (never trained on): {test_countries}")

    model = YOLO(args.model)
    t0 = time.time()
    model.train(
        data=str(data_yaml_path),
        epochs=args.epochs,
        imgsz=args.imgsz,
        batch=args.batch,
        seed=args.seed,
        patience=args.patience,
        project=args.project,
        name=args.name,
        exist_ok=True,
        plots=True,
    )
    train_seconds = time.time() - t0
    train_run_dir = Path(args.project) / args.name

    # Evaluate the SAME trained model on the in-domain test split and on each target country, one
    # domain at a time, using the identical model.val(..., split="test") code path for all of them so
    # the numbers are comparable - the only thing that changes between calls is which dataset.yaml
    # (and therefore which images) is passed in.
    eval_project = Path(args.project) / "eval"
    domains = {}  # domain name -> {"overall":..., "per_class":..., "n_images":..., "run_dir":...}

    print(f"\nEvaluating on in-domain test split ({split_config['n_in_domain_test']} images, "
          f"never seen during training)...")
    in_domain_results = model.val(data=str(data_yaml_path), split="test", plots=True,
                                   project=str(eval_project), name="in_domain", exist_ok=True)
    overall, per_class = _summarize_box(in_domain_results.box, class_names)
    domains["in_domain"] = {"overall": overall, "per_class": per_class,
                             "n_images": split_config["n_in_domain_test"],
                             "run_dir": str(in_domain_results.save_dir)}
    print(f"  in_domain: mAP50={overall['map50']:.4f} mAP50-95={overall['map50_95']:.4f}")

    for country in test_countries:
        eval_yaml_path = split_dir / f"eval_{country}.yaml"
        n_images = split_config["target_country_counts"].get(country, "?")
        print(f"\nEvaluating on target country {country} ({n_images} images, never trained on)...")
        results = model.val(data=str(eval_yaml_path), split="test", plots=True,
                             project=str(eval_project), name=country, exist_ok=True)
        overall, per_class = _summarize_box(results.box, class_names)
        domains[country] = {"overall": overall, "per_class": per_class, "n_images": n_images,
                             "run_dir": str(results.save_dir)}
        print(f"  {country}: mAP50={overall['map50']:.4f} mAP50-95={overall['map50_95']:.4f}")

    in_domain_map50 = domains["in_domain"]["overall"]["map50"]

    metrics = {
        "model": args.model,
        "seed": args.seed,
        "epochs": args.epochs,
        "imgsz": args.imgsz,
        "batch": args.batch,
        "train_seconds": round(train_seconds, 1),
        "split_config": split_config,
        "domains": domains,
        "train_run_dir": str(train_run_dir),
    }
    metrics_path = eval_project / "cross_country_metrics.json"
    eval_project.mkdir(parents=True, exist_ok=True)
    metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")

    report_lines = [
        "# Phase 3 cross-country baseline",
        "",
        f"Generated by `scripts/train_cross_country_baseline.py`. One model trained on "
        f"{split_config['train_countries']}, evaluated separately on the in-domain held-out test "
        f"split and on each of {test_countries} - none of which the model ever trained on.",
        "",
        "## Config (for reproducibility)",
        "",
        f"- model: `{args.model}`",
        f"- seed: {args.seed}",
        f"- epochs: {args.epochs} (patience={args.patience})",
        f"- imgsz: {args.imgsz}, batch: {args.batch}",
        f"- train time: {train_seconds / 60:.1f} min",
        f"- source (train) countries: {split_config['train_countries']} - "
        f"train={split_config['n_train']} val={split_config['n_val']} "
        f"in_domain_test={split_config['n_in_domain_test']} "
        f"(split seed={split_config['seed']})",
        "",
        "## In-domain vs. cross-domain comparison",
        "",
        "The in-domain row is the fair baseline to compare every target country against - same model,"
        " same held-out discipline, only difference is whether the country was in the training mix.",
        "",
        "| domain | n images | mAP@50 | mAP@50-95 | precision | recall | Δ mAP@50 vs in-domain |",
        "|---|---|---|---|---|---|---|",
    ]
    for domain_name, d in domains.items():
        o = d["overall"]
        delta = "-" if domain_name == "in_domain" else f"{o['map50'] - in_domain_map50:+.4f}"
        report_lines.append(
            f"| {domain_name} | {d['n_images']} | {o['map50']:.4f} | {o['map50_95']:.4f} | "
            f"{o['precision_mean']:.4f} | {o['recall_mean']:.4f} | {delta} |"
        )

    report_lines += ["", "## Per-class breakdown by domain", ""]
    for domain_name, d in domains.items():
        report_lines += [
            f"### {domain_name}",
            "",
            "| class | precision | recall | AP50 | AP50-95 |",
            "|---|---|---|---|---|",
        ]
        for c in d["per_class"]:
            if c["precision"] is None:
                report_lines.append(f"| {c['class']} | - | - | - | - ({c.get('note', '')}) |")
            else:
                report_lines.append(f"| {c['class']} | {c['precision']:.4f} | {c['recall']:.4f} | "
                                     f"{c['ap50']:.4f} | {c['ap50_95']:.4f} |")
        report_lines.append("")

    report_lines += ["## Artifacts", ""]
    for domain_name, d in domains.items():
        run_dir = Path(d["run_dir"])
        report_lines.append(
            f"- **{domain_name}**: confusion matrix `{run_dir / 'confusion_matrix.png'}`, "
            f"PR/F1/P/R curves `{run_dir}/Box{{PR,F1,P,R}}_curve.png`"
        )
    report_lines += [
        f"- training curves (loss/mAP per epoch): `{train_run_dir / 'results.png'}`",
        f"- machine-readable metrics: `{metrics_path}`",
        "",
    ]
    report_path = eval_project / "cross_country_report.md"
    report_path.write_text("\n".join(report_lines) + "\n")

    print(f"\nWrote {metrics_path}")
    print(f"Wrote {report_path}")
    print(f"\nSummary (mAP@50): in_domain={in_domain_map50:.4f}", end="")
    for country in test_countries:
        m = domains[country]["overall"]["map50"]
        print(f", {country}={m:.4f} ({m - in_domain_map50:+.4f})", end="")
    print()
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile scripts/diagnose_failures.py
"""Sample and visualize concrete misdetections per target country, for phase 3's qualitative
failure-mode diagnosis (roadmap: infra-defect-detection, phase 3 - distribution-shift diagnosis).

WHY THIS EXISTS: `train_cross_country_baseline.py` answers "how much worse" (a number per country).
This script answers "worse HOW" - a diagnostic report needs concrete failure images, not just an
mAP drop, to argue convincingly for WHICH cross-country factor (lighting/weather, camera
resolution/equipment, road-material appearance, or annotation-standard differences between
countries) is actually driving the gap, and to design a TARGETED mitigation instead of a generic one.

For every image in a target country's held-out eval set, this script runs the trained model,
matches its predictions against ground truth by IoU + class, and classifies each ground-truth box as
a true positive, a missed detection (false negative), or a wrong-class match, and each prediction
with no matching ground-truth region as a false positive. It then saves annotated images (ground
truth in blue, predictions in red) for the worst-scoring images per country - the ones most worth
looking at by eye - plus a machine-readable per-image breakdown and a markdown report.

This script does NOT itself conclude which distribution-shift factor is responsible - that's a
judgment call for a human looking at the saved images - it produces the evidence (ranked failure
cases + per-image brightness, as a cheap proxy for lighting differences worth cross-referencing
against data/eda_report.md's per-country brightness numbers from phase 1) that judgment is based on.

Usage:
    python scripts/diagnose_failures.py --weights runs/phase3/cross_country_baseline/weights/best.pt \\
        --split-dir data/splits/cross_country
    python scripts/diagnose_failures.py --weights <best.pt> --split-dir <dir> \\
        --countries Norway China_Drone --top-n 8 --conf 0.25 --iou 0.5
"""
import argparse
import json
import sys
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont, ImageStat
from ultralytics import YOLO

DEFAULT_OUT_DIR = Path(__file__).resolve().parent.parent / "runs" / "phase3" / "failures"

GT_COLOR = (30, 100, 255)     # blue - ground truth
PRED_COLOR = (255, 40, 40)    # red - model prediction


def _label_path_for(image_path: Path) -> Path:
    # Ultralytics convention: last 'images' path component -> 'labels', extension -> .txt.
    parts = list(image_path.parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"
            break
    return Path(*parts).with_suffix(".txt")


def _read_gt_boxes(label_path: Path, img_w: int, img_h: int):
    """Returns list of (cls, x1, y1, x2, y2) in pixel coords."""
    if not label_path.exists():
        return []
    boxes = []
    for line in label_path.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        cls, cx, cy, w, h = line.split()[:5]
        cls, cx, cy, w, h = int(cls), float(cx), float(cy), float(w), float(h)
        x1, y1 = (cx - w / 2) * img_w, (cy - h / 2) * img_h
        x2, y2 = (cx + w / 2) * img_w, (cy + h / 2) * img_h
        boxes.append((cls, x1, y1, x2, y2))
    return boxes


def _iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def _match(gt_boxes, pred_boxes, iou_thresh):
    """gt_boxes/pred_boxes: list of (cls, x1, y1, x2, y2); pred_boxes assumed sorted by confidence
    descending. Returns (tp, fn, fp, wrong_class) counts and per-box match info for drawing."""
    gt_matched = [False] * len(gt_boxes)
    tp = fn = fp = wrong_class = 0
    pred_status = []  # "tp" | "wrong_class" | "fp", parallel to pred_boxes
    for p_cls, *p_box in pred_boxes:
        best_iou, best_idx = 0.0, -1
        for i, (g_cls, *g_box) in enumerate(gt_boxes):
            if gt_matched[i]:
                continue
            iou = _iou(p_box, g_box)
            if iou > best_iou:
                best_iou, best_idx = iou, i
        if best_idx >= 0 and best_iou >= iou_thresh:
            gt_matched[best_idx] = True
            if gt_boxes[best_idx][0] == p_cls:
                tp += 1
                pred_status.append("tp")
            else:
                wrong_class += 1
                pred_status.append("wrong_class")
        else:
            fp += 1
            pred_status.append("fp")
    fn = sum(1 for m in gt_matched if not m)
    return tp, fn, fp, wrong_class, gt_matched, pred_status


def _draw_annotated(image_path, gt_boxes, pred_boxes, gt_matched, pred_status, class_names, out_path, banner):
    img = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.load_default()
    except Exception:
        font = None

    for (cls, x1, y1, x2, y2), matched in zip(gt_boxes, gt_matched):
        draw.rectangle([x1, y1, x2, y2], outline=GT_COLOR, width=2)
        label = f"GT:{class_names[cls]}" + ("" if matched else " (MISSED)")
        draw.text((x1 + 2, max(0, y1 - 10)), label, fill=GT_COLOR, font=font)

    for (cls, x1, y1, x2, y2, conf), status in zip(pred_boxes, pred_status):
        draw.rectangle([x1, y1, x2, y2], outline=PRED_COLOR, width=2)
        tag = {"tp": "", "wrong_class": " (WRONG CLASS)", "fp": " (FALSE POS)"}[status]
        draw.text((x1 + 2, y2 + 2), f"pred:{class_names[cls]} {conf:.2f}{tag}", fill=PRED_COLOR, font=font)

    # Banner: pad a strip at the top with the failure summary so the image is self-explanatory
    # without needing to cross-reference the JSON/report.
    banner_h = 18
    banner_img = Image.new("RGB", (img.width, img.height + banner_h), color=(255, 255, 255))
    banner_img.paste(img, (0, banner_h))
    ImageDraw.Draw(banner_img).text((2, 2), banner, fill=(0, 0, 0), font=font)
    banner_img.save(out_path, "JPEG")


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--weights", required=True, help="path to a trained .pt checkpoint (e.g. runs/phase3/cross_country_baseline/weights/best.pt)")
    parser.add_argument("--split-dir", required=True, help="dir written by make_cross_country_split.py (has split_config.json + eval_<country>.txt)")
    parser.add_argument("--countries", nargs="+", default=None, help="target countries to diagnose (default: all test_countries in split_config.json)")
    parser.add_argument("--top-n", type=int, default=6, help="number of worst-scoring images to save per country")
    parser.add_argument("--conf", type=float, default=0.25, help="prediction confidence threshold")
    parser.add_argument("--iou", type=float, default=0.5, help="IoU threshold for matching a prediction to a ground-truth box")
    parser.add_argument("--out-dir", default=str(DEFAULT_OUT_DIR))
    args = parser.parse_args(argv)

    weights_path = Path(args.weights)
    if not weights_path.exists():
        print(f"{weights_path} not found - run train_cross_country_baseline.py first.", file=sys.stderr)
        return 1

    split_dir = Path(args.split_dir)
    split_config_path = split_dir / "split_config.json"
    if not split_config_path.exists():
        print(f"{split_config_path} not found - {split_dir} doesn't look like a make_cross_country_split.py output dir.", file=sys.stderr)
        return 1
    split_config = json.loads(split_config_path.read_text())
    countries = args.countries or split_config["test_countries"]

    missing = [c for c in countries if not (split_dir / f"eval_{c}.txt").exists()]
    if missing:
        print(f"No eval_<country>.txt for {missing} in {split_dir}.", file=sys.stderr)
        return 1

    model = YOLO(str(weights_path))
    class_names = model.names if isinstance(model.names, list) else [model.names[i] for i in sorted(model.names)]

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    country_summaries = {}
    for country in countries:
        image_paths = [Path(p) for p in (split_dir / f"eval_{country}.txt").read_text().splitlines() if p.strip()]
        print(f"\n{country}: diagnosing {len(image_paths)} images (conf>={args.conf}, iou>={args.iou})...")

        per_image = []
        for image_path in image_paths:
            if not image_path.exists():
                print(f"  WARNING: {image_path} not found, skipping", file=sys.stderr)
                continue
            with Image.open(image_path) as im:
                img_w, img_h = im.size
                brightness = ImageStat.Stat(im.convert("L")).mean[0]

            gt_boxes = _read_gt_boxes(_label_path_for(image_path), img_w, img_h)

            result = model.predict(source=str(image_path), conf=args.conf, verbose=False)[0]
            pred_boxes = []
            for box in result.boxes:
                x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
                pred_boxes.append((int(box.cls[0]), x1, y1, x2, y2, float(box.conf[0])))
            pred_boxes.sort(key=lambda b: -b[5])
            pred_boxes_for_match = [(b[0], b[1], b[2], b[3], b[4]) for b in pred_boxes]

            tp, fn, fp, wrong_class, gt_matched, pred_status = _match(gt_boxes, pred_boxes_for_match, args.iou)
            failure_score = fn + fp + wrong_class

            per_image.append({
                "image": str(image_path), "n_gt": len(gt_boxes), "n_pred": len(pred_boxes),
                "tp": tp, "fn": fn, "fp": fp, "wrong_class": wrong_class,
                "failure_score": failure_score, "brightness": round(brightness, 1),
                "_gt_boxes": gt_boxes, "_pred_boxes": pred_boxes,
                "_gt_matched": gt_matched, "_pred_status": pred_status,
            })

        per_image.sort(key=lambda r: -r["failure_score"])
        top = per_image[:args.top_n]

        country_out_dir = out_dir / country
        country_out_dir.mkdir(parents=True, exist_ok=True)
        saved = []
        for rank, r in enumerate(top, start=1):
            image_path = Path(r["image"])
            banner = (f"{country} | {image_path.name} | TP={r['tp']} FN={r['fn']} FP={r['fp']} "
                      f"wrong_class={r['wrong_class']} | brightness={r['brightness']}")
            out_path = country_out_dir / f"rank{rank:02d}_score{r['failure_score']}_{image_path.stem}.jpg"
            _draw_annotated(image_path, r["_gt_boxes"], r["_pred_boxes"], r["_gt_matched"],
                             r["_pred_status"], class_names, out_path, banner)
            saved.append(str(out_path))
            print(f"  rank{rank}: {image_path.name} (failure_score={r['failure_score']}, "
                  f"TP={r['tp']} FN={r['fn']} FP={r['fp']} wrong_class={r['wrong_class']}) -> {out_path}")

        totals = {
            "n_images": len(per_image),
            "total_gt": sum(r["n_gt"] for r in per_image),
            "total_tp": sum(r["tp"] for r in per_image),
            "total_fn": sum(r["fn"] for r in per_image),
            "total_fp": sum(r["fp"] for r in per_image),
            "total_wrong_class": sum(r["wrong_class"] for r in per_image),
            "mean_failure_score": round(sum(r["failure_score"] for r in per_image) / len(per_image), 2) if per_image else 0.0,
            "mean_brightness": round(sum(r["brightness"] for r in per_image) / len(per_image), 1) if per_image else 0.0,
        }
        country_summaries[country] = {
            "totals": totals,
            "top_images": [{k: v for k, v in r.items() if not k.startswith("_")} | {"annotated_path": p}
                           for r, p in zip(top, saved)],
        }
        print(f"  {country} totals: TP={totals['total_tp']} FN={totals['total_fn']} "
              f"FP={totals['total_fp']} wrong_class={totals['total_wrong_class']} "
              f"(recall~{totals['total_tp']/(totals['total_tp']+totals['total_fn']):.2f})"
              if totals["total_tp"] + totals["total_fn"] > 0 else "")

    summary_path = out_dir / "failures_summary.json"
    summary_path.write_text(json.dumps({"conf": args.conf, "iou": args.iou, "countries": country_summaries}, indent=2) + "\n")

    report_lines = [
        "# Phase 3 failure-mode diagnosis",
        "",
        f"Generated by `scripts/diagnose_failures.py` from `{weights_path}` (conf>={args.conf}, "
        f"iou>={args.iou}). Ground truth boxes are blue, model predictions are red, in every "
        f"annotated image below - annotated images are saved under `{out_dir}/<country>/`.",
        "",
        "## Country severity summary",
        "",
        "| country | images | GT boxes | TP | FN (missed) | FP (extra) | wrong-class | recall | mean brightness |",
        "|---|---|---|---|---|---|---|---|---|",
    ]
    for country, s in country_summaries.items():
        t = s["totals"]
        recall = t["total_tp"] / (t["total_tp"] + t["total_fn"]) if (t["total_tp"] + t["total_fn"]) > 0 else float("nan")
        report_lines.append(
            f"| {country} | {t['n_images']} | {t['total_gt']} | {t['total_tp']} | {t['total_fn']} | "
            f"{t['total_fp']} | {t['total_wrong_class']} | {recall:.2f} | {t['mean_brightness']} |"
        )

    report_lines += ["", "## Worst-scoring images per country", "",
                      "Cross-reference `mean brightness` here against each country's brightness in "
                      "`data/eda_report.md` (phase 1) - a target country noticeably darker/brighter "
                      "than the source countries is evidence for a lighting-driven gap; if brightness "
                      "is similar but FN/FP is still high, look at the images themselves for "
                      "resolution, road-material, or annotation-style differences instead.", ""]
    for country, s in country_summaries.items():
        report_lines.append(f"### {country}")
        report_lines.append("")
        for img in s["top_images"]:
            report_lines.append(
                f"- `{Path(img['image']).name}` - TP={img['tp']} FN={img['fn']} FP={img['fp']} "
                f"wrong_class={img['wrong_class']} brightness={img['brightness']} -> "
                f"`{img['annotated_path']}`"
            )
        report_lines.append("")
    report_path = out_dir / "failures_report.md"
    report_path.write_text("\n".join(report_lines) + "\n")

    print(f"\nWrote {summary_path}")
    print(f"Wrote {report_path}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 1. 安装依赖

In [ ]:
!pip install -q ultralytics pandas Pillow pyyaml
import torch
print("CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (go enable the GPU accelerator in Session options!)")


## 2. 下载 + MD5校验

`download_rdd2022.py`会先检查上面挂载的`rdd2022-figshare-zip-cache` Dataset里有没有全部7个国家已经解压好的数据——有的话直接打印一下路径就结束,什么都不下载。没挂载这个Dataset(或者Dataset里缺了某个国家)的话,才会退回到原来的路子:12.35GB的合并包,MD5校验通过才继续(和phase 2一样)。

In [ ]:
!python scripts/download_rdd2022.py --skip-extract

## 3. 解压转换全部7个国家

和上一步一样,`extract_convert_per_country.py`会先检查`/kaggle/input/`下有没有这个缓存Dataset——有就直接从里面7个国家的解压好的原始数据转换成YOLO格式(不涉及任何zip操作,几秒钟就能跑完全部7个国家)。没有的话才会走原来那条路:不加`--countries`筛选,默认全部7个国家(Japan/India/Czech/Norway/United_States/China_MotorBike/China_Drone——正好是phase 3 source+target需要的全部国家),合并zip读完所有国家的字节后立刻删除,不会同时占用"zip+解压后数据"两份磁盘空间——这是`extract_convert_per_country.py`专门为Kaggle 19.5GiB配额设计的读法,这次是第一次全量跑,数据量从phase 2的1,072张(Czech一国)变成23,767张(全部7国),没有缓存的话预计比phase 2那一步慢不少。

In [ ]:
!python scripts/extract_convert_per_country.py

## 4. 生成跨国split

source三国(Japan/India/Czech)合并后按70/15/15切train/val/in_domain_test(固定种子42,跨国家打乱,保证每个split里三国都有代表性,不会出现"train全是Japan"这种情况)。target四国(Norway/United_States/China_MotorBike/China_Drone)各自生成一份全量eval list——模型完全没训练过这些国家的任何一张图,所以每一张都能拿来评测,不需要再切分。

In [ ]:
!python scripts/make_cross_country_split.py --seed 42

## 5. 训练 + 跨国评测

**训练时间预估**:source三国合并后约12,195张图(vs phase 2 Czech单国1,072张,约11倍),100 epochs在T4上大概率要2-2.5小时(phase 2跑1,072张、100 epochs花了13.4分钟)。`patience=20`会在20轮没提升时提前停,实际可能更快。**这一步跑起来后可以先去做别的事,不用一直盯着**——上次phase 2也是这样,notebook会一直在后台跑,回来看结果就行。

训练完,同一个模型(不重新训练)依次在:in-domain held-out test(训练国家里切出来的、没训练过的部分)→ Norway → United_States → China_MotorBike → China_Drone,五个集合上分别跑`model.val()`,用完全一样的评测代码路径,保证数字可比。

In [ ]:
!python scripts/train_cross_country_baseline.py \
    --data data/splits/cross_country/dataset.yaml \
    --model yolo11n.pt \
    --epochs 100 \
    --imgsz 640 \
    --batch 16 \
    --seed 42


## 6. 失败案例诊断

针对每个target国家,把训练好的模型跑一遍该国全部图片,和ground truth按IoU+类别匹配,统计每张图的漏检(FN)/多检(FP)/分类错误(wrong-class),挑失败最严重的几张画出标注图(GT蓝框,预测红框),给phase 3的诊断报告用——光看mAP掉了多少不够,得看具体是漏检还是分类错、亮度是不是明显不一样。

In [ ]:
!python scripts/diagnose_failures.py \
    --weights runs/phase3/cross_country_baseline/weights/best.pt \
    --split-dir data/splits/cross_country \
    --top-n 6 \
    --conf 0.25 \
    --iou 0.5


## 7. 核对结果

In [ ]:
!echo '--- split_config.json ---'
!cat data/splits/cross_country/split_config.json
!echo
!echo '--- cross_country_report.md ---'
!find runs -name cross_country_report.md -exec cat {} \;
!echo
!echo '--- failures_report.md ---'
!find runs -name failures_report.md -exec cat {} \;
!echo
!echo '--- 产出文件树 ---'
!find runs data/splits/cross_country -type f | sort


## 8. 收尾:打包成一个zip一次性下载

phase 2下11个文件太费劲,这次直接把所有产出打成一个zip,右侧Output文件树里只用下载这一个文件。

In [ ]:
!zip -r phase3_deliverables.zip \
    data/splits/cross_country \
    runs/phase3 \
    -x "*.cache"
!ls -lh phase3_deliverables.zip


下载好`phase3_deliverables.zip`后,和phase 2一样告诉我一声,我来同步到Mac仓库并做MD5校验。